<a href="https://colab.research.google.com/github/delaneydow/ncl_diffusion/blob/main/00_setup_and_package.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Setup & package install
Run this notebook **once per project** (re-run to update the code). It
1. mounts Drive and creates the persistent project folder,
2. installs dependencies (MONAI Generative Models + pydicom + kaggle),
3. writes the `hemaug` Python package (one cell per module, so you can edit any module in place and re-run that cell),
4. writes a default `config.json` that every other notebook reads.

**Architecture** (every box is swappable through a registry + the config):
```
hemaug/
  config.py            dataclass configs + dotted overrides for ablations
  registry.py          DATASETS / BACKBONES / CONDITIONERS / HEADS / CLASSIFIERS / AUGMENTATIONS / LABEL_PLANS
  data/   base.py (DatasetAdapter)  rsna.py (Kaggle+DICOM)  cache.py  splits.py (patient-level)  datasets.py  augment.py
  models/ backbones.py  conditioners.py  heads.py (aux cls/seg on UNet features)  classifiers.py
          system.py    DiffusionSystem = conditioner + backbone + scheduler + feature hooks + aux heads, CFG sampling
  train/  diffusion.py (AMP, EMA, resume)  classifier.py  ema.py
  synth.py (label plans + generation)   metrics.py (FID, memorization check)   experiments.py (sweeps)
```
Notebook flow: `01` data → `02` train diffusion → `03` generate + evaluate synthetic data → `04` augmentation experiments → `05` extension recipes.

In [ ]:
import os, sys, importlib.util, subprocess
IN_COLAB = "google.colab" in sys.modules
PROJECT = "/content/drive/MyDrive/hemaug" if IN_COLAB else os.path.abspath("hemaug_project")   # <-- persistent project folder (edit if you like)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(PROJECT, exist_ok=True)
LINK = "/content/hemaug_proj" if IN_COLAB else PROJECT
if LINK != PROJECT and not os.path.exists(LINK):
    os.symlink(PROJECT, LINK)
_pip = {"monai-generative": "generative", "pydicom": "pydicom", "kaggle": "kaggle", "scikit-learn": "sklearn",
        "tqdm": "tqdm", "opencv-python-headless": "cv2", "pandas": "pandas", "matplotlib": "matplotlib"}
_missing = [p for p, m in _pip.items() if importlib.util.find_spec(m) is None]
if _missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
if LINK not in sys.path:
    sys.path.insert(0, LINK)
%load_ext autoreload
%autoreload 2

## Package files

In [ ]:
import os
for d in ["hemaug", "hemaug/data", "hemaug/models", "hemaug/train"]:
    os.makedirs(f"{LINK}/{d}", exist_ok=True)

In [ ]:
%%writefile /content/hemaug_proj/hemaug/__init__.py
"""hemaug: modular diffusion-based data-augmentation testbed (RSNA ICH first, other datasets later)."""
__version__ = "0.1.0"

In [ ]:
%%writefile /content/hemaug_proj/hemaug/compat.py
"""Single import point for MONAI generative components.
Uses the `generative` package (Project-MONAI/GenerativeModels); falls back to MONAI core if the
components have been upstreamed in your installed version."""
try:
    from generative.networks.nets import DiffusionModelUNet
    from generative.networks.schedulers import DDPMScheduler, DDIMScheduler
    SOURCE = "monai-generative"
except ImportError:  # pragma: no cover
    from monai.networks.nets import DiffusionModelUNet
    from monai.networks.schedulers import DDPMScheduler, DDIMScheduler
    SOURCE = "monai-core"

In [ ]:
%%writefile /content/hemaug_proj/hemaug/config.py
"""All hyper-parameters live here. Every notebook loads/saves the same config.json in the project folder.
Dotted overrides make ablations one-liners:  cfg2 = cfg.override({"diffusion.num_channels": [32,64,128]})"""
from __future__ import annotations
import copy, json
from dataclasses import dataclass, field, asdict
from pathlib import Path

RSNA_LABELS = ["epidural", "intraparenchymal", "intraventricular", "subarachnoid", "subdural", "any"]


@dataclass
class PathsCfg:
    project: str = "/content/hemaug_proj"   # symlink to Drive project dir (persistent)
    raw: str = "/content/raw"               # ephemeral local disk for raw DICOMs (fast)


@dataclass
class DataCfg:
    dataset: str = "rsna_ich"
    competition: str = "rsna-intracranial-hemorrhage-detection"
    n_images: int = 6000              # subset size (full dataset is far too big for Colab)
    pos_fraction: float | None = 0.35 # fraction with strat_col==1 in subset (None = natural prevalence)
    img_size: int = 128
    windows: list = field(default_factory=lambda: [[40, 80]])  # [[center,width],...] -> channels. brain=(40,80) subdural=(80,200) bone=(600,2800)
    label_cols: list = field(default_factory=lambda: list(RSNA_LABELS))
    strat_col: str = "any"
    group_col: str = "patient_id"     # splits are done at this level to avoid leakage
    split_fracs: list = field(default_factory=lambda: [0.7, 0.1, 0.2])  # train / val / test
    seed: int = 42
    batch_size: int = 32
    num_workers: int = 2
    download_workers: int = 8


@dataclass
class DiffusionCfg:
    backbone: str = "monai_unet"
    conditional: bool = True          # label-conditional (cross-attention) + classifier-free guidance
    conditioner: str = "multihot_mlp"
    cond_dim: int = 256
    cond_tokens: int = 1
    cond_drop: float = 0.1            # prob. of replacing labels by the learned null token (for CFG)
    num_channels: list = field(default_factory=lambda: [64, 128, 256])
    attention_levels: list = field(default_factory=lambda: [False, False, True])
    num_head_channels: list = field(default_factory=lambda: [0, 0, 64])
    num_res_blocks: int = 2
    norm_num_groups: int = 32
    num_train_timesteps: int = 1000
    schedule: str = "scaled_linear_beta"
    beta_start: float = 0.0015
    beta_end: float = 0.0195
    prediction_type: str = "epsilon"  # or "v_prediction"
    # --- extension points (empty by default) ---
    feature_hooks: dict = field(default_factory=dict)  # {"mid": "middle_block", "down2": "down_blocks.2"}
    aux_heads: dict = field(default_factory=dict)      # {"cls": {"type":"cls_pool","feature":"mid","in_channels":256,"out_dim":6,"target":"label","weight":0.1}}
    aux_t_max: int = 250              # aux losses only on samples with t < aux_t_max (features are meaningful at low noise)


@dataclass
class TrainCfg:
    run_name: str = "ddpm_base"
    epochs: int = 40
    lr: float = 1e-4
    weight_decay: float = 0.0
    amp: bool = True
    ema_decay: float = 0.999
    grad_clip: float = 1.0
    ckpt_every: int = 2
    sample_every: int = 5
    n_preview: int = 16


@dataclass
class SampleCfg:
    sampler: str = "ddim"
    steps: int = 50
    cfg_scale: float = 2.0
    batch_size: int = 64
    label_plan: str = "balance"       # match_dist | balance | positive_only
    n_synth: int = 8000
    seed: int = 123


@dataclass
class ClfCfg:
    arch: str = "densenet121"
    epochs: int = 12
    lr: float = 1e-4
    weight_decay: float = 1e-4
    batch_size: int = 64
    augmentation: str = "classical"   # none | classical
    amp: bool = True
    seeds: list = field(default_factory=lambda: [0, 1, 2])


_SECTIONS = ("paths", "data", "diffusion", "train", "sample", "clf")


@dataclass
class Config:
    paths: PathsCfg = field(default_factory=PathsCfg)
    data: DataCfg = field(default_factory=DataCfg)
    diffusion: DiffusionCfg = field(default_factory=DiffusionCfg)
    train: TrainCfg = field(default_factory=TrainCfg)
    sample: SampleCfg = field(default_factory=SampleCfg)
    clf: ClfCfg = field(default_factory=ClfCfg)

    # ---- paths ----
    def project_dir(self, *p) -> Path:
        return Path(self.paths.project).joinpath(*p)

    def cache_dir(self) -> Path:
        d = self.data
        return self.project_dir("cache", f"{d.dataset}_{d.img_size}px_n{d.n_images}_w{len(d.windows)}")

    def run_dir(self, name=None) -> Path:
        return self.project_dir("runs", name or self.train.run_name)

    # ---- (de)serialisation ----
    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_dict(cls, d):
        cfg = cls()
        for sec in _SECTIONS:
            for k, v in d.get(sec, {}).items():
                setattr(getattr(cfg, sec), k, v)
        return cfg

    def save(self, path=None):
        path = Path(path or self.project_dir("config.json"))
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(self.to_dict(), indent=2))
        return path

    @classmethod
    def load(cls, path):
        return cls.from_dict(json.loads(Path(path).read_text()))

    def override(self, updates: dict):
        """Return a deep copy with dotted-key updates applied, e.g. {"diffusion.cond_drop": 0.2}."""
        new = copy.deepcopy(self)
        for key, v in updates.items():
            sec, name = key.split(".", 1)
            sect = getattr(new, sec)
            if not hasattr(sect, name):
                raise AttributeError(f"{key} is not a config field")
            setattr(sect, name, v)
        return new

In [ ]:
%%writefile /content/hemaug_proj/hemaug/experiments.py
"""Downstream augmentation experiments: train classifier on real (+ synthetic) and evaluate on held-out data."""
import itertools
from pathlib import Path
from torch.utils.data import ConcatDataset
import pandas as pd
from .data import make_dataset
from .registry import AUGMENTATIONS
from .train.classifier import train_classifier, evaluate
from .utils import seed_everything, get_device


def run_experiment(cfg, images, manifest, synth=None, ratio=0.0, seed=0, aug=None, arch=None, device=None, tag=""):
    device = device or get_device()
    cfg = cfg.override({"clf.arch": arch or cfg.clf.arch, "clf.augmentation": aug or cfg.clf.augmentation})
    lc, in_ch = cfg.data.label_cols, images.shape[1]
    seed_everything(seed)
    tf = AUGMENTATIONS.build(cfg.clf.augmentation, cfg.data.img_size)
    parts = [make_dataset(images, manifest, "train", lc, tf)]
    n_real, n_syn = len(parts[0]), 0
    if ratio > 0:
        s_img, s_man = synth
        n_syn = min(int(round(ratio * n_real)), len(s_man))
        if n_syn < int(round(ratio * n_real)): print(f"[warn] only {len(s_man)} synthetic images available (wanted {int(round(ratio*n_real))})")
        parts.append(make_dataset(s_img, s_man.iloc[:n_syn], None, lc, tf, is_synth=True))
    val, test = make_dataset(images, manifest, "val", lc), make_dataset(images, manifest, "test", lc)
    model, hist = train_classifier(cfg, ConcatDataset(parts), val, in_ch, device, seed)
    v, t = evaluate(model, val, lc, device), evaluate(model, test, lc, device)
    return {"tag": tag, "arch": cfg.clf.arch, "aug": cfg.clf.augmentation, "synth_ratio": ratio, "seed": seed,
            "n_real": n_real, "n_synth": n_syn, "val_weighted_auc": v["weighted_auc"],
            "test_weighted_auc": t["weighted_auc"], "test_macro_auc": t["macro_auc"],
            **{f"test_{k}": x for k, x in t.items() if k.startswith("auc_")}}


def sweep(cfg, images, manifest, synth, grid, results_path, device=None, tag=""):
    """grid e.g. {"aug": ["none","classical"], "ratio": [0,0.5,1,2], "seed": [0,1,2]}. Resumable: finished
    combos found in results_path (csv) are skipped. Returns the full results DataFrame."""
    results_path = Path(results_path); results_path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()
    keys = list(grid)
    colmap = {"aug": "aug", "ratio": "synth_ratio", "seed": "seed", "arch": "arch"}
    for combo in itertools.product(*[grid[k] for k in keys]):
        kw = dict(zip(keys, combo))
        if len(df):
            mask = pd.Series(True, index=df.index)
            for k, v in kw.items(): mask &= (df[colmap[k]] == v)
            mask &= (df["tag"].fillna("") == tag)
            if mask.any():
                continue
        print("running", kw)
        row = run_experiment(cfg, images, manifest, synth, ratio=kw.get("ratio", 0.0), seed=kw.get("seed", 0),
                             aug=kw.get("aug"), arch=kw.get("arch"), device=device, tag=tag)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True); df.to_csv(results_path, index=False)
    return df

In [ ]:
%%writefile /content/hemaug_proj/hemaug/metrics.py
"""Generative-quality checks. FID needs `pip install torchmetrics torch-fidelity` (Inception weights download)."""
import numpy as np
import torch
import torch.nn.functional as F


def fid_score(real_u8, fake_u8, device="cuda", batch=128, max_n=4000):
    from torchmetrics.image.fid import FrechetInceptionDistance
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)

    def feed(arr, real):
        for i in range(0, min(len(arr), max_n), batch):
            x = torch.from_numpy(np.asarray(arr[i:i + batch])).to(device)
            x = x[:, :1].repeat(1, 3, 1, 1) if x.shape[1] == 1 else x[:, :3]
            fid.update(x, real=real)
    feed(real_u8, True); feed(fake_u8, False)
    return float(fid.compute())


@torch.no_grad()
def nearest_real(fake_u8, real_u8, k=1, device="cuda", res=32, max_real=6000):
    """Memorization check: nearest real training image (low-res L2) for every synthetic image.
    Returns (distances (N,k), indices (N,k) into real_u8). Near-zero distances => copying the training set."""
    def prep(a):
        x = torch.from_numpy(np.asarray(a)).float().to(device) / 255
        return F.adaptive_avg_pool2d(x, res).flatten(1)
    R = prep(real_u8[:max_real]); D, I = [], []
    for i in range(0, len(fake_u8), 512):
        d = torch.cdist(prep(fake_u8[i:i + 512]), R); v, j = d.topk(k, largest=False); D.append(v.cpu()); I.append(j.cpu())
    return torch.cat(D).numpy(), torch.cat(I).numpy()

In [ ]:
%%writefile /content/hemaug_proj/hemaug/registry.py
"""Tiny registry so every swappable piece (dataset, backbone, head, ...) is selected by a string in the config."""


class Registry:
    def __init__(self, name):
        self.name, self._d = name, {}

    def register(self, key=None):
        def deco(obj):
            self._d[key or obj.__name__] = obj
            return obj
        return deco

    def get(self, key):
        if key not in self._d:
            raise KeyError(f"'{key}' not in registry '{self.name}'. Available: {sorted(self._d)}")
        return self._d[key]

    def build(self, key, *args, **kwargs):
        return self.get(key)(*args, **kwargs)

    def keys(self):
        return sorted(self._d)

    def __repr__(self):
        return f"Registry({self.name}: {self.keys()})"


DATASETS = Registry("datasets")            # dataset adapters (download / manifest / load)
AUGMENTATIONS = Registry("augmentations")  # classical augmentation pipelines for the classifier
BACKBONES = Registry("backbones")          # diffusion denoiser networks
CONDITIONERS = Registry("conditioners")    # label -> context embedding
HEADS = Registry("heads")                  # auxiliary heads on diffusion-UNet features (cls / seg / ...)
CLASSIFIERS = Registry("classifiers")      # downstream evaluation classifiers
LABEL_PLANS = Registry("label_plans")      # how to choose conditioning labels for synthetic data

In [ ]:
%%writefile /content/hemaug_proj/hemaug/synth.py
"""Synthetic dataset generation: choose conditioning labels (label plans), sample, save as uint8 cache."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from .registry import LABEL_PLANS


@LABEL_PLANS.register("match_dist")
def match_dist(train_man, label_cols, n, seed):
    """Sample label vectors from the real training set (keeps real prevalence and co-occurrence)."""
    return train_man[label_cols].sample(n, replace=True, random_state=seed).to_numpy(np.float32)


@LABEL_PLANS.register("positive_only")
def positive_only(train_man, label_cols, n, seed, any_col="any"):
    return train_man[train_man[any_col] == 1][label_cols].sample(n, replace=True, random_state=seed).to_numpy(np.float32)


@LABEL_PLANS.register("balance")
def balance(train_man, label_cols, n, seed, any_col="any"):
    """Equal number of samples per category (negative + each subtype; rarest subtype wins for multi-label rows)
    -> oversamples rare hemorrhage types, the case where synthetic augmentation should help most."""
    sub = [c for c in label_cols if c != any_col]
    order = train_man[sub].mean().sort_values(ascending=False).index   # common -> rare, rare overwrites last
    cat = np.full(len(train_man), "neg", dtype=object)
    for c in order:
        cat[train_man[c].to_numpy() == 1] = c
    cats = sorted(set(cat)); per = n // len(cats); rng = np.random.RandomState(seed); out = []
    for c in cats:
        pool = train_man[cat == c][label_cols].to_numpy(np.float32)
        out.append(pool[rng.randint(0, len(pool), per)])
    out = np.concatenate(out); rng.shuffle(out)
    return out


@torch.no_grad()
def generate_synthetic(system, labels, sc, label_cols, out_dir, device="cuda", seed=123):
    """labels: (N,L) float array. Writes out_dir/images.npy (uint8), manifest.csv, sampler.json."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    N, C, S = len(labels), system.in_channels, system.img_size
    arr = np.lib.format.open_memmap(out_dir / "images.npy", mode="w+", dtype=np.uint8, shape=(N, C, S, S))
    g = torch.Generator(device=device).manual_seed(seed)
    system.eval().to(device)
    for i in tqdm(range(0, N, sc.batch_size), desc="sampling"):
        lab = torch.from_numpy(labels[i:i + sc.batch_size]).float().to(device)
        x = system.sample(lab, steps=sc.steps, cfg_scale=sc.cfg_scale, sampler=sc.sampler, generator=g)
        arr[i:i + len(lab)] = ((x + 1) / 2 * 255).round().byte().cpu().numpy()
    arr.flush()
    man = pd.DataFrame(labels, columns=label_cols).astype(int)
    man.insert(0, "sample_id", [f"synth_{i}" for i in range(N)]); man["idx"] = np.arange(N)
    man.to_csv(out_dir / "manifest.csv", index=False)
    (out_dir / "sampler.json").write_text(json.dumps(vars(sc), indent=2))
    return out_dir


def load_synth(out_dir):
    out_dir = Path(out_dir)
    return np.load(out_dir / "images.npy"), pd.read_csv(out_dir / "manifest.csv")

In [ ]:
%%writefile /content/hemaug_proj/hemaug/utils.py
import random
import numpy as np
import torch


def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def count_params(m) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def show_grid(x, nrow=8, title=None, save=None, figsize=(10, 10)):
    """x: (N,C,H,W) tensor/array in [0,1]. Shows first channel (or RGB if C>=3)."""
    import matplotlib.pyplot as plt
    from torchvision.utils import make_grid
    x = torch.as_tensor(np.asarray(x)).float()
    x = x[:, :3] if x.shape[1] >= 3 else x[:, :1]
    g = make_grid(x, nrow=nrow, padding=2, pad_value=1.0).permute(1, 2, 0).numpy()
    plt.figure(figsize=figsize); plt.imshow(g, cmap="gray"); plt.axis("off")
    if title: plt.title(title)
    if save: plt.savefig(save, bbox_inches="tight", dpi=120)
    plt.show() if save is None else plt.close()


def to_model_range(x):  # [0,1] -> [-1,1]
    return x * 2 - 1


def to_unit_range(x):   # [-1,1] -> [0,1]
    return (x.clamp(-1, 1) + 1) / 2

In [ ]:
%%writefile /content/hemaug_proj/hemaug/data/__init__.py
from . import rsna  # noqa: F401  (registers adapters; add new dataset modules here)
from .base import DatasetAdapter
from .cache import build_cache, load_cache
from .splits import stratified_subset, make_splits, split_summary
from .datasets import CachedArrayDataset, make_dataset, make_loader
from . import augment  # noqa: F401

In [ ]:
%%writefile /content/hemaug_proj/hemaug/data/augment.py
"""Classical augmentation pipelines (the comparison baseline for synthetic augmentation)."""
import torch
from ..registry import AUGMENTATIONS


class _Plain:
    """MetaTensor -> plain tensor so default collate stays cheap."""
    def __call__(self, x):
        return x.as_tensor() if hasattr(x, "as_tensor") else torch.as_tensor(x)


@AUGMENTATIONS.register("none")
def none(img_size):
    return None


@AUGMENTATIONS.register("classical")
def classical(img_size):
    from monai.transforms import Compose, RandFlip, RandAffine
    return Compose([
        RandFlip(prob=0.5, spatial_axis=1),
        RandAffine(prob=0.6, rotate_range=(0.17,), scale_range=(0.1, 0.1),
                   translate_range=(img_size * 0.05, img_size * 0.05), padding_mode="zeros", mode="bilinear"),
        _Plain(),
    ])

In [ ]:
%%writefile /content/hemaug_proj/hemaug/data/base.py
from abc import ABC, abstractmethod
import numpy as np
import pandas as pd


class DatasetAdapter(ABC):
    """Everything dataset-specific lives in an adapter. To add a dataset: subclass, register with
    @DATASETS.register("name"), import the module in data/__init__.py, set cfg.data.dataset."""
    label_cols: list = []

    def __init__(self, cfg):
        self.cfg = cfg

    @abstractmethod
    def build_manifest(self) -> pd.DataFrame:
        """One row per sample: must have `sample_id` + one column per label (0/1)."""

    @abstractmethod
    def download(self, sample_ids) -> list:
        """Fetch raw files for these sample_ids to cfg.paths.raw. Return list of failed ids."""

    @abstractmethod
    def load(self, row) -> tuple:
        """Return (img float32 (C,H,W) in [0,1] already resized to cfg.data.img_size, meta dict).
        meta must contain `cfg.data.group_col` (e.g. patient_id) so splits can be grouped."""

In [ ]:
%%writefile /content/hemaug_proj/hemaug/data/cache.py
"""Preprocess raw samples once into one uint8 array (N,C,H,W) + manifest.csv on Drive."""
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def build_cache(adapter, manifest, cache_dir, workers=8):
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    manifest = manifest.reset_index(drop=True)
    N, S, C = len(manifest), adapter.cfg.data.img_size, len(adapter.cfg.data.windows)
    arr = np.lib.format.open_memmap(cache_dir / "images.npy", mode="w+", dtype=np.uint8, shape=(N, C, S, S))
    rows = list(manifest.itertuples(index=False))

    def work(i):
        try:
            img, meta = adapter.load(rows[i])
            arr[i] = np.round(img * 255).astype(np.uint8)
            return i, meta, None
        except Exception as e:
            return i, {}, repr(e)

    metas, errs = {}, {}
    with ThreadPoolExecutor(workers) as ex:
        for i, meta, err in tqdm(ex.map(work, range(N)), total=N, desc="preprocess"):
            metas[i] = meta
            if err: errs[i] = err
    arr.flush()
    meta_df = pd.DataFrame([metas[i] for i in range(N)])
    out = pd.concat([manifest, meta_df], axis=1)
    out["idx"] = np.arange(N)                       # row index into images.npy (stable through filtering/splitting)
    out = out[~out.idx.isin(errs)].reset_index(drop=True)
    out.to_csv(cache_dir / "manifest.csv", index=False)
    if errs:
        print(f"{len(errs)} failed samples dropped, e.g. {list(errs.items())[:3]}")
    return out


def load_cache(cache_dir, mmap=False):
    cache_dir = Path(cache_dir)
    images = np.load(cache_dir / "images.npy", mmap_mode="r" if mmap else None)
    return images, pd.read_csv(cache_dir / "manifest.csv")

In [ ]:
%%writefile /content/hemaug_proj/hemaug/data/datasets.py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class CachedArrayDataset(Dataset):
    """Serves images from the uint8 cache as float [0,1] tensors. Used for both real and synthetic data."""
    def __init__(self, images, manifest, label_cols, transform=None, is_synth=False):
        self.images, self.transform, self.is_synth = images, transform, is_synth
        self.idx = manifest["idx"].to_numpy()
        self.labels = manifest[label_cols].to_numpy(np.float32)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        x = torch.from_numpy(np.asarray(self.images[self.idx[i]], dtype=np.float32) / 255.0)
        if self.transform is not None:
            x = self.transform(x)
        return {"image": x, "label": torch.from_numpy(self.labels[i]), "is_synth": torch.tensor(self.is_synth)}


def make_dataset(images, manifest, split, label_cols, transform=None, is_synth=False):
    m = manifest if split is None else manifest[manifest.split == split]
    return CachedArrayDataset(images, m, label_cols, transform, is_synth)


def make_loader(ds, batch_size, shuffle, num_workers=2, drop_last=False):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
                      drop_last=drop_last, pin_memory=torch.cuda.is_available(),
                      persistent_workers=num_workers > 0)

In [ ]:
%%writefile /content/hemaug_proj/hemaug/data/rsna.py
import os, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from ..registry import DATASETS
from .base import DatasetAdapter

RSNA_LABELS = ["epidural", "intraparenchymal", "intraventricular", "subarachnoid", "subdural", "any"]


def kaggle_login():
    """Kaggle creds from env, Colab secrets (KAGGLE_USERNAME / KAGGLE_KEY), or ~/.kaggle/kaggle.json."""
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        except Exception:
            if not Path("~/.kaggle/kaggle.json").expanduser().exists():
                raise RuntimeError("No Kaggle credentials: add Colab secrets KAGGLE_USERNAME/KAGGLE_KEY "
                                   "or upload kaggle.json to ~/.kaggle/")
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    return api


def window_hu(hu, center, width):
    lo = center - width / 2
    return np.clip((hu - lo) / width, 0, 1)


@DATASETS.register("rsna_ich")
class RSNAAdapter(DatasetAdapter):
    label_cols = RSNA_LABELS
    train_dir = "stage_2_train"

    @property
    def raw(self) -> Path:
        return Path(self.cfg.paths.raw) / "rsna"

    # ---------------- Kaggle ----------------
    def _fetch(self, api, relpath, retries=4):
        dest = self.raw / relpath
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists():
            return True
        for a in range(retries):
            try:
                api.competition_download_file(self.cfg.data.competition, relpath, path=str(dest.parent), quiet=True)
                z = dest.with_name(dest.name + ".zip")
                if z.exists():
                    zipfile.ZipFile(z).extractall(dest.parent); z.unlink()
                if dest.exists():
                    return True
            except Exception:  # rate limits / transient errors
                time.sleep(2 ** a)
        return False

    def download_file(self, relpath):
        return self._fetch(kaggle_login(), relpath)

    def download(self, sample_ids):
        api = kaggle_login()
        ids = list(sample_ids)
        with ThreadPoolExecutor(self.cfg.data.download_workers) as ex:
            ok = list(tqdm(ex.map(lambda s: self._fetch(api, f"{self.train_dir}/{s}.dcm"), ids), total=len(ids), desc="download"))
        return [s for s, o in zip(ids, ok) if not o]

    # ---------------- manifest ----------------
    def build_manifest(self):
        csv = self.raw / "stage_2_train.csv"
        if not csv.exists():
            assert self.download_file("stage_2_train.csv"), "could not download stage_2_train.csv (accepted the competition rules?)"
        df = pd.read_csv(csv)
        parts = df["ID"].str.rsplit("_", n=1, expand=True)
        df["sample_id"], df["sub"] = parts[0], parts[1]
        wide = df.pivot_table(index="sample_id", columns="sub", values="Label", aggfunc="first").reset_index()
        return wide[["sample_id"] + self.label_cols].astype({c: int for c in self.label_cols})

    # ---------------- image loading ----------------
    @staticmethod
    def _to_hu(d):
        px = d.pixel_array.astype(np.int32)
        slope, inter = float(getattr(d, "RescaleSlope", 1)), float(getattr(d, "RescaleIntercept", 0))
        # known RSNA quirk: some 12-bit unsigned files have wrapped values / wrong intercept
        if getattr(d, "BitsStored", 16) == 12 and getattr(d, "PixelRepresentation", 0) == 0 and inter > -100:
            px = px + 1000; px[px >= 4096] -= 4096; inter = -1000.0
        return px * slope + inter

    def load(self, row):
        import cv2, pydicom
        d = pydicom.dcmread(self.raw / self.train_dir / f"{row.sample_id}.dcm")
        hu = self._to_hu(d)
        S = self.cfg.data.img_size
        chans = [cv2.resize(window_hu(hu, c, w).astype(np.float32), (S, S), interpolation=cv2.INTER_AREA)
                 for c, w in self.cfg.data.windows]
        meta = {"patient_id": str(d.PatientID), "study_uid": str(getattr(d, "StudyInstanceUID", "")),
                "series_uid": str(getattr(d, "SeriesInstanceUID", ""))}
        return np.stack(chans).astype(np.float32), meta

In [ ]:
%%writefile /content/hemaug_proj/hemaug/data/splits.py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


def stratified_subset(manifest, n, strat_col="any", pos_fraction=0.35, seed=42):
    if n is None or n >= len(manifest):
        return manifest.copy()
    if pos_fraction is None:
        return manifest.sample(n, random_state=seed).reset_index(drop=True)
    pos, neg = manifest[manifest[strat_col] == 1], manifest[manifest[strat_col] == 0]
    n_pos = min(int(round(n * pos_fraction)), len(pos)); n_neg = min(n - n_pos, len(neg))
    out = pd.concat([pos.sample(n_pos, random_state=seed), neg.sample(n_neg, random_state=seed)])
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)


def make_splits(manifest, group_col="patient_id", strat_col="any", fracs=(0.7, 0.1, 0.2), seed=42):
    """Group-level (patient) stratified train/val/test split -> adds a `split` column. No patient crosses splits."""
    tr, va, te = fracs
    assert abs(tr + va + te - 1) < 1e-6
    g = manifest.groupby(group_col)[strat_col].max()          # patient is positive if any slice is
    pats, y = g.index.to_numpy(), g.to_numpy()
    p_trva, p_te = train_test_split(pats, test_size=te, stratify=y, random_state=seed)
    y_trva = g.loc[p_trva].to_numpy()
    p_tr, p_va = train_test_split(p_trva, test_size=va / (tr + va), stratify=y_trva, random_state=seed)
    mapping = {**{p: "train" for p in p_tr}, **{p: "val" for p in p_va}, **{p: "test" for p in p_te}}
    out = manifest.copy(); out["split"] = out[group_col].map(mapping)
    sets = {s: set(out.loc[out.split == s, group_col]) for s in ("train", "val", "test")}
    assert not (sets["train"] & sets["val"] or sets["train"] & sets["test"] or sets["val"] & sets["test"]), "patient leakage!"
    return out


def split_summary(manifest, label_cols, group_col="patient_id"):
    rows = []
    for s in ("train", "val", "test"):
        m = manifest[manifest.split == s]
        r = {"split": s, "n_images": len(m), "n_patients": m[group_col].nunique()}
        r.update({f"prev_{c}": round(m[c].mean(), 3) for c in label_cols})
        rows.append(r)
    return pd.DataFrame(rows).set_index("split")

In [ ]:
%%writefile /content/hemaug_proj/hemaug/models/__init__.py
from . import conditioners, backbones, heads, classifiers  # noqa: F401  (register components)
from .system import DiffusionSystem, FeatureRecorder, build_scheduler

In [ ]:
%%writefile /content/hemaug_proj/hemaug/models/backbones.py
"""Denoiser backbones. Contract: builder(d, in_channels, cross_attention_dim) -> nn.Module with
forward(x, timesteps, context=None) -> tensor like x. Register new ones with @BACKBONES.register("name")."""
from ..registry import BACKBONES
from ..compat import DiffusionModelUNet


@BACKBONES.register("monai_unet")
def monai_unet(d, in_channels, cross_attention_dim=None):
    return DiffusionModelUNet(
        spatial_dims=2, in_channels=in_channels, out_channels=in_channels,
        num_res_blocks=d.num_res_blocks,
        num_channels=tuple(d.num_channels),
        attention_levels=tuple(d.attention_levels),
        num_head_channels=tuple(d.num_head_channels),
        norm_num_groups=d.norm_num_groups,
        with_conditioning=cross_attention_dim is not None,
        cross_attention_dim=cross_attention_dim,
    )

In [ ]:
%%writefile /content/hemaug_proj/hemaug/models/classifiers.py
"""Downstream classifiers used to measure the augmentation effect. builder(in_channels, n_labels) -> nn.Module."""
from ..registry import CLASSIFIERS


@CLASSIFIERS.register("densenet121")
def densenet121(in_channels, n_labels):
    from monai.networks.nets import DenseNet121
    return DenseNet121(spatial_dims=2, in_channels=in_channels, out_channels=n_labels)


@CLASSIFIERS.register("resnet18")
def resnet18(in_channels, n_labels):
    from monai.networks.nets import resnet18 as _r18
    return _r18(spatial_dims=2, n_input_channels=in_channels, num_classes=n_labels)

In [ ]:
%%writefile /content/hemaug_proj/hemaug/models/conditioners.py
import torch
import torch.nn as nn
from ..registry import CONDITIONERS


@CONDITIONERS.register("multihot_mlp")
class MultiHotConditioner(nn.Module):
    """multi-hot label vector -> (B, n_tokens, dim) context for cross-attention, plus a learned
    null token used for classifier-free-guidance dropout."""
    def __init__(self, n_labels, dim=256, n_tokens=1, hidden=512):
        super().__init__()
        self.dim, self.n_tokens = dim, n_tokens
        self.net = nn.Sequential(nn.Linear(n_labels, hidden), nn.SiLU(), nn.Linear(hidden, dim * n_tokens))
        self.null = nn.Parameter(torch.randn(1, n_tokens, dim) * 0.02)

    def forward(self, labels, drop_mask=None):
        c = self.net(labels).view(labels.shape[0], self.n_tokens, self.dim)
        if drop_mask is not None:
            c = torch.where(drop_mask.view(-1, 1, 1), self.null.expand_as(c).to(c.dtype), c)
        return c

    def null_context(self, n):
        return self.null.expand(n, -1, -1)

In [ ]:
%%writefile /content/hemaug_proj/hemaug/models/heads.py
"""Auxiliary heads that read intermediate diffusion-UNet features (captured by FeatureRecorder) and add a
loss to training. Contract: forward(feat)->out ; loss(out, target)->(B,) per-sample loss.
Enable via cfg.diffusion.feature_hooks + cfg.diffusion.aux_heads (see notebook 05)."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from ..registry import HEADS


@HEADS.register("cls_pool")
class PooledClassificationHead(nn.Module):
    """Multi-label classification from pooled features (e.g. the UNet mid-block)."""
    def __init__(self, in_channels, out_dim, hidden=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_channels, hidden),
                                 nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, out_dim))

    def forward(self, feat):
        return self.net(feat)

    def loss(self, out, target):
        return F.binary_cross_entropy_with_logits(out.float(), target.float(), reduction="none").mean(1)


@HEADS.register("seg_upsample")
class SegmentationHead(nn.Module):
    """Template segmentation head: 1x1 conv + bilinear upsample to mask size. (RSNA has no masks; use with
    a dataset adapter that provides batch['mask'] of shape (B,1,H,W).)"""
    def __init__(self, in_channels, out_channels=1, size=128):
        super().__init__()
        self.conv, self.size = nn.Conv2d(in_channels, out_channels, 1), size

    def forward(self, feat):
        return F.interpolate(self.conv(feat), size=(self.size, self.size), mode="bilinear", align_corners=False)

    def loss(self, out, target):
        p = torch.sigmoid(out.float()); t = target.float()
        bce = F.binary_cross_entropy_with_logits(out.float(), t, reduction="none").mean((1, 2, 3))
        dice = 1 - (2 * (p * t).sum((1, 2, 3)) + 1) / (p.sum((1, 2, 3)) + t.sum((1, 2, 3)) + 1)
        return bce + dice

In [ ]:
%%writefile /content/hemaug_proj/hemaug/models/system.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from ..registry import BACKBONES, CONDITIONERS, HEADS
from ..compat import DDPMScheduler, DDIMScheduler


def build_scheduler(d, kind="ddpm"):
    cls = {"ddpm": DDPMScheduler, "ddim": DDIMScheduler}[kind]
    return cls(num_train_timesteps=d.num_train_timesteps, schedule=d.schedule,
               beta_start=d.beta_start, beta_end=d.beta_end, prediction_type=d.prediction_type)


class FeatureRecorder:
    """Forward hooks that stash intermediate activations by name. spec = {name: dotted module path}."""
    def __init__(self, module, spec):
        self.store, self.handles = {}, []
        for name, path in (spec or {}).items():
            self.handles.append(module.get_submodule(path).register_forward_hook(self._make(name)))

    def _make(self, name):
        def hook(_, __, out):
            self.store[name] = out[0] if isinstance(out, (tuple, list)) else out
        return hook

    def clear(self):
        self.store.clear()

    def remove(self):
        for h in self.handles: h.remove()


class DiffusionSystem(nn.Module):
    """conditioner + denoiser backbone + noise scheduler + optional feature hooks/aux heads, all built from
    config through registries. Swap/ablate any piece by editing the config or registering a new component."""
    def __init__(self, cfg, in_channels, n_labels):
        super().__init__()
        d = cfg.diffusion
        self.d, self.in_channels, self.img_size = d, in_channels, cfg.data.img_size
        self.conditioner = (CONDITIONERS.build(d.conditioner, n_labels=n_labels, dim=d.cond_dim, n_tokens=d.cond_tokens)
                            if d.conditional else None)
        self.unet = BACKBONES.build(d.backbone, d, in_channels, d.cond_dim if d.conditional else None)
        self.scheduler = build_scheduler(d, "ddpm")
        self.recorder = FeatureRecorder(self.unet, d.feature_hooks)
        self.heads, self.head_specs = nn.ModuleDict(), {}
        for name, spec in d.aux_heads.items():
            spec = dict(spec)
            self.head_specs[name] = dict(feature=spec.pop("feature"), target=spec.pop("target", "label"),
                                         weight=spec.pop("weight", 0.1))
            self.heads[name] = HEADS.build(spec.pop("type"), **spec)

    @property
    def device(self):
        return next(self.parameters()).device

    # ------------------------------------------------------------------ training
    def training_step(self, batch):
        """batch: dict with 'image' in [-1,1], 'label' (B,L) and optionally 'mask'. Returns (loss, logs)."""
        x, d = batch["image"], self.d
        B = x.shape[0]
        noise = torch.randn_like(x)
        t = torch.randint(0, d.num_train_timesteps, (B,), device=x.device).long()
        x_t = self.scheduler.add_noise(x, noise, t)
        ctx = None
        if self.conditioner is not None:
            drop = (torch.rand(B, device=x.device) < d.cond_drop) if self.training else None
            ctx = self.conditioner(batch["label"], drop)
        self.recorder.clear()
        pred = self.unet(x_t, t, context=ctx)
        target = noise if d.prediction_type == "epsilon" else self.scheduler.get_velocity(x, noise, t)
        loss = F.mse_loss(pred.float(), target.float())
        logs = {"diffusion": loss.item()}
        low_t = (t < d.aux_t_max).float()
        for name, head in self.heads.items():
            sp = self.head_specs[name]
            per = head.loss(head(self.recorder.store[sp["feature"]]), batch[sp["target"]])
            l_aux = (per * low_t).sum() / low_t.sum().clamp(min=1)
            loss = loss + sp["weight"] * l_aux
            logs[name] = l_aux.item()
        logs["total"] = loss.item()
        return loss, logs

    # ------------------------------------------------------------------ sampling
    @torch.no_grad()
    def sample(self, labels=None, n=None, steps=50, cfg_scale=2.0, sampler="ddim", generator=None, amp=True):
        dev = self.device
        n = labels.shape[0] if labels is not None else n
        sched = build_scheduler(self.d, sampler); sched.set_timesteps(steps)
        x = torch.randn(n, self.in_channels, self.img_size, self.img_size, device=dev, generator=generator)
        use_cfg = self.conditioner is not None and cfg_scale != 1.0
        if self.conditioner is not None:
            ctx = self.conditioner(labels.to(dev).float())
            null = self.conditioner.null_context(n)
        for t in sched.timesteps:
            tt = torch.full((n,), int(t), device=dev, dtype=torch.long)
            with torch.autocast(dev.type, dtype=torch.float16, enabled=amp and dev.type == "cuda"):
                if use_cfg:
                    out = self.unet(torch.cat([x, x]), torch.cat([tt, tt]), context=torch.cat([null, ctx]))
                else:
                    out = self.unet(x, tt, context=ctx if self.conditioner is not None else None)
            out = out.float()
            if use_cfg:
                u, c = out.chunk(2)
                out = u + cfg_scale * (c - u)
            x, _ = sched.step(out, int(t), x)
        return x.clamp(-1, 1)   # [-1,1]

In [ ]:
%%writefile /content/hemaug_proj/hemaug/train/__init__.py
from .ema import EMA
from .diffusion import train_diffusion, load_system
from .classifier import train_classifier, evaluate, predict

In [ ]:
%%writefile /content/hemaug_proj/hemaug/train/classifier.py
import copy
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from ..registry import CLASSIFIERS
from ..utils import to_model_range


@torch.no_grad()
def predict(model, ds, device, bs=128):
    model.eval(); P, Y = [], []
    for b in DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2):
        P.append(torch.sigmoid(model(to_model_range(b["image"]).to(device)).float()).cpu()); Y.append(b["label"])
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def evaluate(model, ds, label_cols, device):
    P, Y = predict(model, ds, device)
    aucs = {c: (roc_auc_score(Y[:, i], P[:, i]) if len(np.unique(Y[:, i])) > 1 else float("nan")) for i, c in enumerate(label_cols)}
    w = np.array([2.0 if c == "any" else 1.0 for c in label_cols])   # RSNA-style: 'any' counts double
    v = np.array([aucs[c] for c in label_cols]); ok = ~np.isnan(v)
    return {"macro_auc": float(np.nanmean(v)), "weighted_auc": float((v[ok] * w[ok]).sum() / w[ok].sum()), **{f"auc_{c}": a for c, a in aucs.items()}}


def train_classifier(cfg, train_ds, val_ds, in_channels, device, seed=0):
    cc, lc = cfg.clf, cfg.data.label_cols
    model = CLASSIFIERS.build(cc.arch, in_channels, len(lc)).to(device)
    loader = DataLoader(train_ds, batch_size=cc.batch_size, shuffle=True, num_workers=cfg.data.num_workers,
                        drop_last=True, generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=cc.lr, weight_decay=cc.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cc.epochs * len(loader))
    use_amp = cc.amp and device == "cuda"
    S = getattr(torch.amp, "GradScaler", None)
    scaler = S("cuda", enabled=use_amp) if S else torch.cuda.amp.GradScaler(enabled=use_amp)
    best, best_state, hist = -1, None, []
    for ep in range(cc.epochs):
        model.train(); losses = []
        for b in loader:
            x, y = to_model_range(b["image"]).to(device), b["label"].to(device)
            with torch.autocast(device, dtype=torch.float16, enabled=use_amp):
                loss = F.binary_cross_entropy_with_logits(model(x).float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        m = evaluate(model, val_ds, lc, device)
        hist.append({"epoch": ep + 1, "train_loss": float(np.mean(losses)), "val_weighted_auc": m["weighted_auc"]})
        if m["weighted_auc"] > best:
            best, best_state = m["weighted_auc"], copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, hist

In [ ]:
%%writefile /content/hemaug_proj/hemaug/train/diffusion.py
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from ..utils import to_model_range, to_unit_range, show_grid
from .ema import EMA


def _scaler(enabled):
    S = getattr(torch.amp, "GradScaler", None)
    return S("cuda", enabled=enabled) if S else torch.cuda.amp.GradScaler(enabled=enabled)


def _prep(batch, device):
    b = {k: v.to(device, non_blocking=True) for k, v in batch.items() if torch.is_tensor(v)}
    b["image"] = to_model_range(b["image"])
    return b


@torch.no_grad()
def validate(system, loader, device, amp):
    system.eval(); tot, n = 0.0, 0
    with torch.random.fork_rng():
        torch.manual_seed(0)   # fixed noise/timesteps -> comparable val loss across epochs
        for batch in loader:
            b = _prep(batch, device)
            with torch.autocast(device, dtype=torch.float16, enabled=amp and device == "cuda"):
                _, logs = system.training_step(b)
            tot += logs["diffusion"] * len(b["image"]); n += len(b["image"])
    system.train()
    return tot / n


def train_diffusion(system, train_loader, val_loader, cfg, run_dir, device="cuda", preview_labels=None):
    tc, run_dir = cfg.train, Path(run_dir)
    (run_dir / "samples").mkdir(parents=True, exist_ok=True)
    system.to(device)
    opt = torch.optim.AdamW(system.parameters(), lr=tc.lr, weight_decay=tc.weight_decay)
    use_amp = tc.amp and device == "cuda"
    scaler, ema = _scaler(use_amp), EMA(system, tc.ema_decay)
    start, history, best = 0, [], float("inf")
    last = run_dir / "ckpt_last.pt"
    if last.exists():
        ck = torch.load(last, map_location=device, weights_only=False)
        system.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); ema.load_state_dict(ck["ema"])
        start, history, best = ck["epoch"] + 1, ck["history"], ck["best"]
        print(f"resumed from epoch {start}")

    for ep in range(start, tc.epochs):
        system.train(); run = {}
        pbar = tqdm(train_loader, desc=f"epoch {ep+1}/{tc.epochs}", leave=False)
        for batch in pbar:
            b = _prep(batch, device)
            with torch.autocast(device, dtype=torch.float16, enabled=use_amp):
                loss, logs = system.training_step(b)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(system.parameters(), tc.grad_clip)
            scaler.step(opt); scaler.update(); ema.update(system)
            for k, v in logs.items(): run.setdefault(k, []).append(v)
            pbar.set_postfix(loss=f"{logs['total']:.4f}")
        row = {"epoch": ep + 1, **{f"train_{k}": float(np.mean(v)) for k, v in run.items()}}
        row["val_diffusion"] = validate(system, val_loader, device, tc.amp)
        history.append(row); print({k: round(v, 5) for k, v in row.items()})

        if row["val_diffusion"] < best:
            best = row["val_diffusion"]
            torch.save({"model": system.state_dict(), "ema": ema.shadow, "epoch": ep, "val": best}, run_dir / "ckpt_best.pt")
        if (ep + 1) % tc.ckpt_every == 0 or ep + 1 == tc.epochs:
            torch.save({"model": system.state_dict(), "opt": opt.state_dict(), "ema": ema.state_dict(),
                        "epoch": ep, "history": history, "best": best}, last)
        if preview_labels is not None and ((ep + 1) % tc.sample_every == 0 or ep + 1 == tc.epochs):
            system.eval()
            with ema.average_parameters(system):
                x = system.sample(torch.as_tensor(preview_labels[:tc.n_preview]).float().to(device), steps=50,
                                  cfg_scale=cfg.sample.cfg_scale, generator=torch.Generator(device).manual_seed(0))
            show_grid(to_unit_range(x).cpu(), nrow=8, title=f"EMA samples, epoch {ep+1}",
                      save=run_dir / "samples" / f"ep{ep+1:03d}.png"); system.train()
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    return pd.DataFrame(history)


def load_system(cfg, run_dir, in_channels, n_labels, device="cuda", which="best", use_ema=True):
    """Rebuild the system from cfg and load EMA (or raw) weights from ckpt_{which}.pt."""
    from ..models import DiffusionSystem
    system = DiffusionSystem(cfg, in_channels, n_labels)
    ck = torch.load(Path(run_dir) / f"ckpt_{which}.pt", map_location="cpu", weights_only=False)
    if use_ema:
        sd = ck["ema"]["shadow"] if "shadow" in ck["ema"] else ck["ema"]
    else:
        sd = ck["model"]
    system.load_state_dict(sd)
    return system.to(device).eval()

In [ ]:
%%writefile /content/hemaug_proj/hemaug/train/ema.py
from contextlib import contextmanager
import torch


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay, self.n = decay, 0
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point: self.shadow[k].mul_(d).add_(v.detach(), alpha=1 - d)
            else: self.shadow[k].copy_(v)

    @contextmanager
    def average_parameters(self, model):
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)
        try: yield
        finally: model.load_state_dict(backup)

    def state_dict(self): return {"decay": self.decay, "n": self.n, "shadow": self.shadow}
    def load_state_dict(self, s): self.decay, self.n, self.shadow = s["decay"], s["n"], s["shadow"]

## Default config
Edit the values here (or override later in any notebook) and save. **Colab T4 tip:** if you hit OOM in notebook 02, lower `data.batch_size` or `diffusion.num_channels`.

In [ ]:
from hemaug.config import Config
cfg = Config()
cfg.paths.project = LINK
# cfg.data.n_images = 6000; cfg.data.img_size = 128
# cfg.data.windows = [[40, 80]]                       # 1 channel (brain window)
# cfg.data.windows = [[40, 80], [80, 200], [600, 2800]]  # 3 channels: brain/subdural/bone
print(cfg.save())

In [ ]:
import torch, hemaug
from hemaug import compat
from hemaug.registry import BACKBONES, DATASETS, CLASSIFIERS
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch runtime to GPU)")
print("generative components from:", compat.SOURCE)
print(DATASETS, BACKBONES, CLASSIFIERS, sep="\n")